In [6]:
import time
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_scipy_sparse_matrix, add_self_loops, degree, normalized_cut
from torch_sparse import SparseTensor

# --- 1. Load Cora dataset ---
dataset = Planetoid(root="data/Planetoid", name="Cora")
data = dataset[0]
num_nodes = data.num_nodes
num_features = dataset.num_node_features
num_classes = dataset.num_classes

def random_edge_sampling(edge_index, q):
    num_edges = edge_index.shape[1]
    sampled_indices = torch.randperm(num_edges)[:q]
    sampled_edge_index = edge_index[:, sampled_indices]
    return sampled_edge_index

edge_index = data.edge_index

q = int(data.edge_index.shape[1]*0.2)
edge_index = random_edge_sampling(data.edge_index,q)



# --- 2. Build normalized adjacency matrix (A_hat) as a PyTorch sparse tensor ---
# Add self-loops to edge_index
edge_index_with_loops, _ = add_self_loops(edge_index, num_nodes=num_nodes)

# Compute degrees
row, col = edge_index_with_loops
deg = degree(col, num_nodes=num_nodes, dtype=torch.float32)  # degree of each node (including self‐loops)
deg_inv_sqrt = deg.pow(-0.5)
deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0

# Compute normalization: for each edge i->j, weight = 1 / sqrt(deg[i] * deg[j])
norm_values = deg_inv_sqrt[row] * deg_inv_sqrt[col]

# Create a sparse tensor A_hat in COO format
A_hat = torch.sparse_coo_tensor(
    indices=torch.stack([row, col], dim=0),
    values=norm_values,
    size=(num_nodes, num_nodes)
).coalesce().to(device='cpu')  # keep on CPU or move to GPU later if desired

# --- 3. Define a simple two-layer GCN using manual spmm and ddmm steps ---
class ManualGCN(torch.nn.Module):
    def __init__(self, in_feats, hidden_feats, out_feats):
        super().__init__()
        self.W1 = torch.nn.Parameter(torch.randn(in_feats, hidden_feats))
        self.W2 = torch.nn.Parameter(torch.randn(hidden_feats, out_feats))
        # Initialize weights (optional but recommended)
        torch.nn.init.xavier_uniform_(self.W1)
        torch.nn.init.xavier_uniform_(self.W2)

    def forward(self, x, A_sparse):
        # Layer 1: X' = ReLU( A_hat @ X @ W1 )
        #  3.1 Sparse‐Dense multiply: support1 = A_hat @ X
        support1 = torch.sparse.mm(A_sparse, x)
        #  3.2 Dense‐Dense multiply: hidden = support1 @ W1
        hidden = support1 @ self.W1
        hidden = F.relu(hidden)

        # Layer 2: X'' = A_hat @ hidden @ W2, followed by log_softmax
        support2 = torch.sparse.mm(A_sparse, hidden)
        out_logits = support2 @ self.W2
        return F.log_softmax(out_logits, dim=1)

# --- 4. Set up timing instrumentation ---
# Global accumulators for time spent in different categories:
time_spmm = 0.0      # sparse‐dense matrix multiply
time_ddmm = 0.0      # dense‐dense matrix multiply
time_total = 0.0     # total forward time

# Keep references to the original functions
_original_sparse_mm = torch.sparse.mm
_original_matmul = torch.Tensor.__matmul__  # the "@" operator is Tensor.__matmul__

# Define wrapped versions
def _timed_sparse_mm(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    global time_spmm
    t0 = time.time()
    res = _original_sparse_mm(a, b)
    t1 = time.time()
    time_spmm += (t1 - t0)
    return res

def _timed_matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    global time_ddmm
    t0 = time.time()
    res = _original_matmul(a, b)
    t1 = time.time()
    time_ddmm += (t1 - t0)
    return res

# Monkey‐patch the functions
torch.sparse.mm = _timed_sparse_mm
torch.Tensor.__matmul__ = _timed_matmul

# --- 5. Prepare model, data, and run forward passes to accumulate timings ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ManualGCN(num_features, 16, num_classes).to(device)

# Move data and adjacency to the same device
x = data.x.to(device)
A_hat = A_hat.to(device)

# Run several forward passes (e.g., 50) to get stable timing proportions
num_runs = 50
model.eval()
with torch.no_grad():
    for _ in range(num_runs):
        t_start = time.time()
        _ = model(x, A_hat)
        t_end = time.time()
        time_total += (t_end - t_start)

# Restore original functions (optional, if further code should be unaffected)
torch.sparse.mm = _original_sparse_mm
torch.Tensor.__matmul__ = _original_matmul

# --- 6. Compute proportions ---
avg_total = time_total / num_runs
avg_spmm = time_spmm / num_runs
avg_ddmm = time_ddmm / num_runs
avg_others = avg_total - (avg_spmm + avg_ddmm)

print(f"==== Timing over {num_runs} forward passes (Cora) ====")
print(f"Average total forward time:      {avg_total:.6f} s")
print(f" · Sparse‐Dense matmul (spmm):   {avg_spmm:.6f} s ({100.0 * avg_spmm / avg_total:.2f}%)")
print(f" · Dense‐Dense matmul (ddmm):    {avg_ddmm:.6f} s ({100.0 * avg_ddmm / avg_total:.2f}%)")
print(f" · Other operations:             {avg_others:.6f} s ({100.0 * avg_others / avg_total:.2f}%)")


==== Timing over 50 forward passes (Cora) ====
Average total forward time:      0.000183 s
 · Sparse‐Dense matmul (spmm):   0.000114 s (62.15%)
 · Dense‐Dense matmul (ddmm):    0.000037 s (20.17%)
 · Other operations:             0.000032 s (17.68%)
